# Agents Can Act

In the last notebook, we gave an agent **eyes** to see images.

Now let's give it **hands** to interact with a computer.

## Browser Automation

An agent that can:
- Navigate to websites
- Search for products
- Click buttons
- Fill forms
- Take screenshots

In [1]:
from agents import Agent
from agents.mcp import MCPServerStdio
from omniagents import Runner
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
print("Environment loaded!")

Environment loaded!


## The Tools (via Playwright MCP)

| Tool | Action |
|------|--------|
| `browser_navigate` | Go to a URL |
| `browser_click` | Click elements |
| `browser_type` | Type into fields |
| `browser_screenshot` | Capture the screen |
| `browser_snapshot` | Read page structure |

In [2]:
# Configure the Playwright MCP server (headed mode by default)
playwright_server = MCPServerStdio(
    params={
        "command": "npx",
        "args": [
            "@playwright/mcp@latest",
            "--executable-path", "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
            "--isolated",
            "--viewport-size", "1280x720",
        ],
    },
    name="Playwright Browser",
    client_session_timeout_seconds=60,
)


## Create the Shopping Agent

In [3]:
INSTRUCTIONS = """
You are a helpful shopping assistant with the ability to browse the web using a real browser.

## Your Capabilities
- Navigate to retail websites (Target, H-E-B, Walmart, Amazon, etc.)
- Search for products
- Check in-store availability at specific locations
- Check shipping/delivery options
- Add items to cart
- Take screenshots to show what you find

## How to Approach Shopping Tasks

1. **Navigate** to the store's website
2. **Search** for the requested product
3. **Check availability** - look for in-store pickup options at the specified location, or shipping options
4. **Take action** - add to cart if requested
5. **Screenshot** key steps so the user can see what you did

## Tips
- Use the store's location/ZIP code features to check local inventory
- If a product isn't available in-store, check if shipping is an option
- Be specific about what you find - prices, availability status, delivery estimates
- If you encounter popups or modals, try to dismiss them to continue
- Take screenshots at key moments to show your progress

## Important
- Never enter payment information
- Stop at the cart - don't proceed to checkout
- If asked to log in, inform the user and stop
"""

shopping_agent = Agent(
    name="Shopping Assistant",
    instructions=INSTRUCTIONS,
    tools=[],  # MCP server provides the tools
    model="gpt-4.1",
)

## Example 1: Find a Riftbound Deck at Target

Can the agent find a product, check local inventory, and add it to cart?

In [ ]:
runner = Runner.from_agent(shopping_agent, mcp_servers=[playwright_server])
runner.run_notebook(
    input="""Check if Target has a Riftbound Jinx deck available.

First check if it's available for in-store pickup at:
Target - 7400 N 10th St, McAllen, TX

If not available in-store, check if it's available for shipping.
If it is available (either way), add it to my cart."""
)

Error getting response: Error code: 429 - {'error': {'message': 'litellm.RateLimitError: AzureException RateLimitError - {\n  "error": {\n    "message": "Too Many Requests",\n    "type": "too_many_requests",\n    "param": null,\n    "code": "too_many_requests"\n  }\n}. Received Model Group=gpt-4.1\nAvailable Model Group Fallbacks=None', 'type': 'throttling_error', 'param': None, 'code': '429'}}
API request failed: Error code: 429 - {'error': {'message': 'litellm.RateLimitError: AzureException RateLimitError - {\n  "error": {\n    "message": "Too Many Requests",\n    "type": "too_many_requests",\n    "param": null,\n    "code": "too_many_requests"\n  }\n}. Received Model Group=gpt-4.1\nAvailable Model Group Fallbacks=None', 'type': 'throttling_error', 'param': None, 'code': '429'}}
Traceback (most recent call last):
  File "/Users/snpr/Code/AI-Agents-Spring26/.venv/lib/python3.13/site-packages/omniagents/core/runtime/bridge.py", line 501, in stream_agent
    async for event in result.st

## Example 2: Grocery Shopping at H-E-B

Can the agent handle a multi-item shopping list with specific requirements?

In [5]:
runner = Runner.from_agent(shopping_agent, mcp_servers=[playwright_server])
runner.run_notebook(
    input="""I need to pick up a few things from H-E-B for a cookout.

Store location: H-E-B on Freddy Gonzalez Dr in Edinburg, TX

Please find and add to cart:
1. Fajita meat - prime grade preferred, no more than 2 lbs
2. Charcoal - a small bag of Kingsford

Check availability for curbside pickup at that location."""
)

Error getting response: Error code: 429 - {'error': {'message': 'litellm.RateLimitError: AzureException RateLimitError - {\n  "error": {\n    "message": "Too Many Requests",\n    "type": "too_many_requests",\n    "param": null,\n    "code": "too_many_requests"\n  }\n}. Received Model Group=gpt-4.1\nAvailable Model Group Fallbacks=None', 'type': 'throttling_error', 'param': None, 'code': '429'}}
API request failed: Error code: 429 - {'error': {'message': 'litellm.RateLimitError: AzureException RateLimitError - {\n  "error": {\n    "message": "Too Many Requests",\n    "type": "too_many_requests",\n    "param": null,\n    "code": "too_many_requests"\n  }\n}. Received Model Group=gpt-4.1\nAvailable Model Group Fallbacks=None', 'type': 'throttling_error', 'param': None, 'code': '429'}}
Traceback (most recent call last):
  File "/Users/snpr/Code/AI-Agents-Spring26/.venv/lib/python3.13/site-packages/omniagents/core/runtime/bridge.py", line 501, in stream_agent
    async for event in result.st

## The Agent Loop (Browser Edition)

| Step | What Happens |
|------|--------------|
| 1 | "Find Riftbound at Target" |
| 2 | `browser_navigate` → target.com |
| 3 | `browser_type` → search "Riftbound" |
| 4 | `browser_click` → search button |
| 5 | `browser_snapshot` → read results |
| 6 | `browser_click` → "Add to cart" |
| 7 | `browser_screenshot` → proof |

The agent decides what to click and when.

## Try It Yourself

In [ ]:
runner = Runner.from_agent(shopping_agent, mcp_servers=[playwright_server])
runner.run_notebook()

## What Else Could It Do?

Same pattern, different websites:
- Check stock at Best Buy
- Compare prices on Amazon
- Book a restaurant reservation
- Fill out forms
- Automate any repetitive web task

**If you can do it in a browser, an agent can too.**

## Summary

| Notebook | Agent Capability |
|----------|-----------------|
| **101** | Tools (call functions) |
| **102** | Vision (see images) |
| **103** | Browser (click, type, navigate) |

**Perception + Action = Agents in the real world**